In [ ]:
# Import polars and set configurations
# *** do not try to open things without "head"***

import polars as pl
_=pl.Config.set_tbl_cols(100000)
_=pl.Config.set_tbl_rows(10000)
_=pl.Config.set_tbl_width_chars(10000)
_=pl.Config.set_fmt_str_lengths(10000)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Volcano Plots for RBPs Bound at Pos X

# RBPs with the greatest absolute local mean shap (bound only)

## Repressors (neg. shap)
Position 2:
- KHDRBS1 (-0.177) K562

Position 3:
- HNRNPC (-0.316) HEPG2
- SUGP2 (-0.275) HEPG2
- ILF3 (-0.219) HEPG2
- SAFB (-0.217) K562
- SFPQ (-0.149) HEPG2
- NKRF (-0.142) HEPG2
- MATR3 (-0.136) HEPG2
- SAFB2 (-0.134) K562

Position 4:
- SUGP2 (-0.314) HEPG2
- ILF3 (-0.171) HEPG2

Position 5:
- NCBP2 (-0.269) HEPG2
- DDX3X (-0.224) K562
- DDX3X (-0.178) HEPG2

Position 6:
- DDX3X (-0.176) K562

## Activators (pos. shap)
Position 3:
- DDX3X (0.059) K562
- IGF2BP1 (0.065) HEPG2
- DDX3X (0.064) HEPG2
- G3BP1 (0.068) HEPG2
- BCLAF1 (0.066) HEPG2

Position 4:
- LIN28B (0.060) HEPG2
- SND1 (0.061) HEPG2
- YBX3 (0.061) HEPG2
- PRPF8 (0.061) HEPG2
- PPIG (0.062) HEPG2
- LARP4 (0.062) HEPG2
- IGF2BP1 (0.064) HEPG2
- G3BP1 (0.065) HEPG2
- BCLAF1 (0.066) HEPG2

Position 5:
- DDX52 (0.070) HEPG2

Position 6:

In [ ]:
# General for every plot

# Columns 
columns_to_keep = [
    "RBP_KD_Target",
    "Sample Name",
    "FDR",
    "DeltaPSI",
    "has_RBP_KD_4",
    "Raw P-Val",
    "rMATS Event ID"
]

# Plot colors

def assign_color(row):
    if row["FDR"] >= 0.05 or abs(row["DeltaPSI"]) < 0.05:
        return "gray"
    elif row["DeltaPSI"] >= 0.05:
        return "red"
    elif row["DeltaPSI"] <= -0.05:
        return "blue"
    else:
        return "gray"  # fallback


In [ ]:
K562_PATH = "/project/PlatigLab/users/yogi/backups/2025-08-28_probability_SHAP_with_all_training_UBPs_as_background/FINAL_AVERAGE_SHAP_CACHE/K562_all-data.feather"
K562 = pl.read_ipc(K562_PATH)

## K562

In [ ]:
# K562

rbps = ["KHDRBS1", "SAFB", "SAFB2", "DDX3X", "DDX3X", "DDX3X"]
positions = [2, 3, 3, 5, 6, 3]
shaps = [-0.177, -0.217, -0.134, -0.224, -0.176, .059]

for rbp, position,  shaps in zip(rbps, positions, shaps):
    
    print(rbp)
    print(position)
    print(shaps)
    
    filtered_K562 = (
    K562
    .filter(
        (pl.col("RBP_KD_Target") == rbp) & (pl.col(f"has_RBP_KD_{position}") == True)
    )
    .unique(subset=["rMATS Event ID"]))
    
    # Filter for needed columns
    filtered_K562 = filtered_K562.select(columns_to_keep)
    
    # Convert to pandas
    df = filtered_K562.to_pandas()
    
    # Prevents error for no data (SAFB or IGF..1)
    data = df["DeltaPSI"]

    if data.empty:
        continue 
    
    # DPSI conversion
    df["DeltaPSI"] = df["DeltaPSI"] * -1
    
    # -log10 FDR conversion
    
    # Find the smallest non-zero FDR value
    min_nonzero_fdr = df.loc[df["FDR"] > 0, "FDR"].min()
    
    # Replace zeros with min
    df["FDR_safe"] = df["FDR"].replace(0, min_nonzero_fdr)

    # Calculate -log10(FDR)
    df["neg_log10_FDR"] = -np.log10(df["FDR_safe"])
    
    # Colors and plots
    
    df["color"] = df.apply(assign_color, axis=1)

    counts = df["color"].value_counts().to_dict()
    num_points = len(df)
    
    # Parameter for global font size
    plt.rcParams.update({'font.size': 15})
    
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(
        df["DeltaPSI"],
        df["neg_log10_FDR"],
        c=df["color"],
        alpha=0.7,
        edgecolors='w',
        linewidth=0.5,
        s = 90
    )
    
    # Setting x axis
    xlim = max(abs(min(df["DeltaPSI"])), abs(max(df["DeltaPSI"])))
    plt.xlim(-xlim - 0.01, xlim + 0.01)  # symmetric around zero
    plt.axvline(0, color="black", linestyle="--")  # reference line at zero

    # Legend
    legend_labels = [
        f"Red (ΔPSI ≥ 0.05): {counts.get('red', 0)}",
        f"Blue (ΔPSI ≤ -0.05): {counts.get('blue', 0)}",
        f"Gray (NS): {counts.get('gray', 0)}",
        f"Total events: {num_points}"
    ]

    # Create dummy handles for legend colors
    from matplotlib.lines import Line2D
    handles = [
        Line2D([0], [0], marker='o', color='w', label=legend_labels[0],
               markerfacecolor='red', markersize=8),
        Line2D([0], [0], marker='o', color='w', label=legend_labels[1],
               markerfacecolor='blue', markersize=8),
        Line2D([0], [0], marker='o', color='w', label=legend_labels[2],
               markerfacecolor='gray', markersize=8),
        Line2D([0], [0], color='w', label=legend_labels[3])  # total count (no color)
    ]

    plt.legend(handles=handles, loc="best")

    # Labels
    plt.xlabel("$\\psi_{ctrl} - \\psi_{KD}$", fontsize = 19, labelpad = 15)
    plt.ylabel("-log10(FDR)", fontsize = 17, labelpad = 20)
    plt.title(f"K562: {rbp} KD Events w/ {rbp} Bound at Pos. {position}", pad=20)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

## HEPG2

In [ ]:
HEPG2_PATH = "/project/PlatigLab/users/yogi/backups/2025-08-28_probability_SHAP_with_all_training_UBPs_as_background/FINAL_AVERAGE_SHAP_CACHE/HepG2_all-data.feather"
HEPG2 = pl.read_ipc(HEPG2_PATH)

In [ ]:
# HepG2

rbps = ["HNRNPC", "SUGP2", "ILF3", "SFPQ", "NKRF", "MATR3", "IGF2BP1", "DDX3X", "G3BP1", "BCLAF1",
        "SUGP2", "ILF3", "LIN28B", "SND1", "YBX3", "PRPF8", "PPIG", "LARP4", "IGF2BP1", "G3BP1", "BCLAF1",
        "NCBP2", "DDX3X", "DDX52"]
        
positions = [3,3,3,3,3,3,3,3,3,3,4,4,4,4,4,4,4,4,4,4,4,5,5,5]

shaps = [-0.316, -0.275, -0.219, -0.149, -0.142, -0.136, 0.065, 0.064, 0.068, 0.066,
        -0.314, -0.171, 0.060, 0.061, 0.061, 0.061, 0.062, 0.062, 0.064, 0.065, 0.066,
        -0.269, -0.178, 0.070]

for rbp, position, shaps in zip(rbps, positions, shaps):
    
    print(rbp)
    print(position)
    
    filtered_HEPG2 = (
    HEPG2
    .filter(
        (pl.col("RBP_KD_Target") == rbp) & (pl.col(f"has_RBP_KD_{position}") == True)
    )
    .unique(subset=["rMATS Event ID"]))
    
    # Filter for needed columns
    filtered_HEPG2 = filtered_HEPG2.select(columns_to_keep)
    
    # Convert to pandas
    df = filtered_HEPG2.to_pandas()
    
    # Prevents error for no data (SAFB or IGF..1)
    data = df["DeltaPSI"]

    if data.empty:
        continue 
    
    # DPSI conversion
    df["DeltaPSI"] = df["DeltaPSI"] * -1
    
    # -log10 FDR conversion
    
    # Find the smallest non-zero FDR value
    min_nonzero_fdr = df.loc[df["FDR"] > 0, "FDR"].min()
    
    # Replace zeros with min
    df["FDR_safe"] = df["FDR"].replace(0, min_nonzero_fdr)

    # Calculate -log10(FDR)
    df["neg_log10_FDR"] = -np.log10(df["FDR_safe"])
    
    # Colors and plots
    
    df["color"] = df.apply(assign_color, axis=1)

    counts = df["color"].value_counts().to_dict()
    num_points = len(df)
    
    # Parameter for global font size
    plt.rcParams.update({'font.size': 15})
    
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(
        df["DeltaPSI"],
        df["neg_log10_FDR"],
        c=df["color"],
        alpha=0.7,
        edgecolors='w',
        linewidth=0.5,
        s = 90
    )
    
    # Setting x axis
    xlim = max(abs(min(df["DeltaPSI"])), abs(max(df["DeltaPSI"])))
    plt.xlim(-xlim - 0.01, xlim + 0.01)  # symmetric around zero
    plt.axvline(0, color="black", linestyle="--")  # reference line at zero

    # Legend
    legend_labels = [
        f"Red (ΔPSI ≥ 0.05): {counts.get('red', 0)}",
        f"Blue (ΔPSI ≤ -0.05): {counts.get('blue', 0)}",
        f"Gray (NS): {counts.get('gray', 0)}",
        f"Total events: {num_points}"
    ]

    # Create dummy handles for legend colors
    from matplotlib.lines import Line2D
    handles = [
        Line2D([0], [0], marker='o', color='w', label=legend_labels[0],
               markerfacecolor='red', markersize=8),
        Line2D([0], [0], marker='o', color='w', label=legend_labels[1],
               markerfacecolor='blue', markersize=8),
        Line2D([0], [0], marker='o', color='w', label=legend_labels[2],
               markerfacecolor='gray', markersize=8),
        Line2D([0], [0], color='w', label=legend_labels[3])  # total count (no color)
    ]

    plt.legend(handles=handles, loc="best")

    # Labels
    plt.xlabel("$\\psi_{ctrl} - \\psi_{KD}$", fontsize = 19, labelpad = 15)
    plt.ylabel("-log10(FDR)", fontsize = 17, labelpad = 20)
    plt.title(f"HepG2: {rbp} KD Events w/ {rbp} Bound at Pos. {position}", pad=20)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()